<a href="https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/notebooks/03_working_with_the_full_release.ipynb)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('xxxxx')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last90,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev90,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last90,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END)       AS pos_last90
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev90 >= 300
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0 content items with enough history


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,pos_last90


In [ ]:
# TASK 2: Add weekend_share feature
weekend_feature = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(CASE WHEN dayofweek(report_date) IN (0, 6) THEN gsc_impressions ELSE 0 END) AS weekend_impressions,
        SUM(gsc_impressions) AS total_impressions,
        SUM(CASE WHEN dayofweek(report_date) IN (0, 6) THEN gsc_impressions ELSE 0 END)
            / NULLIF(SUM(gsc_impressions), 0) AS weekend_share
    FROM {TABLES['fact_daily']}
    GROUP BY 1
    HAVING total_impressions >= 100
""").df()

print(f"{len(weekend_feature):,} rows")
weekend_feature[['content_hash_id', 'weekend_share']].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

182,135 rows


,content_hash_id,weekend_share
0,content_fe8e8155ce1d47a2,0.260355
1,content_b4462a1b90640058,0.302225
2,content_c782fa8abd4fce5e,0.295170
3,content_89c6c2e17e412e20,0.279704
4,content_8a3ccd5a0b61b7f7,0.326117


In [ ]:
# Merge the new feature into `data` (built from features + qsignals in the notebook)
data = data.merge(weekend_feature[['content_hash_id', 'weekend_share']], on='content_hash_id', how='left')
data['weekend_share'] = data['weekend_share'].fillna(0)
print(f"Joined: {len(data):,} rows, weekend_share added")
data.head()

Joined: 111,247 rows, weekend_share added


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,is_declining,weekend_share
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609,2.0,0.037221,0.823821,34.0,56.0,0.607143,1,0.247059
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667,1.0,0.076923,0.446154,93.0,93.0,1.000000,1,0.190955
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574,13.0,0.014321,0.718867,226.0,857.0,0.263711,0,0.264215
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051,2.0,0.129032,0.760753,22.0,41.0,0.536585,1,0.275728
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224,1.0,0.053045,0.843811,105.0,105.0,1.000000,0,0.236686


In [ ]:
# TASK 3: Compare random split vs. client-grouped split
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report

# Label (same definition as the notebook's original Section 5)
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'weekend_share']

model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

# --- Baseline: random split (pages from same client can land in both train and test) ---
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print("=== Random split ===")
print(f"Base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}")
print(classification_report(y_te, model_random.predict(X_te), digits=3))

# --- Honest check: client-grouped split (no client's pages appear in both) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))

X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)

print("\n=== Client-grouped split ===")
print(f"Base rate (always predict majority): {max(y_te_g.mean(), 1 - y_te_g.mean()):.3f}")
print(classification_report(y_te_g, model_grouped.predict(X_te_g), digits=3))

# Sanity check — confirm zero client overlap between train/test
train_clients = set(model_data.iloc[train_idx]['client_hash_id'])
test_clients = set(model_data.iloc[test_idx]['client_hash_id'])
print(f"\nOverlapping clients between train/test: {len(train_clients & test_clients)} (should be 0)")

=== Random split ===
Base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.572     0.350     0.434      9389
           1      0.692     0.848     0.762     16162

    accuracy                          0.665     25551
   macro avg      0.632     0.599     0.598     25551
weighted avg      0.648     0.665     0.642     25551


=== Client-grouped split ===
Base rate (always predict majority): 0.677
              precision    recall  f1-score   support

           0      0.386     0.541     0.450     16706
           1      0.729     0.589     0.651     34996

    accuracy                          0.573     51702
   macro avg      0.557     0.565     0.551     51702
weighted avg      0.618     0.573     0.586     51702


Overlapping clients between train/test: 0 (should be 0)


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609,2.0,0.037221,0.823821,34.0,56.0,0.607143
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667,1.0,0.076923,0.446154,93.0,93.0,1.000000
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574,13.0,0.014321,0.718867,226.0,857.0,0.263711
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051,2.0,0.129032,0.760753,22.0,41.0,0.536585
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224,1.0,0.053045,0.843811,105.0,105.0,1.000000


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.545     0.340     0.419      9389
           1      0.685     0.835     0.753     16162

    accuracy                          0.653     25551
   macro avg      0.615     0.588     0.586     25551
weighted avg      0.634     0.653     0.630     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.
---
# I have done all these here in colab below the cell 3.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples o
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [ ]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address
